# Favorite World voice server - every engine, free Colab

**Aman · Rohit · Divya · Rani · FavoriteWorld (trained) · Chatterbox (Vishal + your clones) · Qwen3-TTS (English) · IndicF5**, all on one link.

1. Use a **free** Google account (a Colab Pro account spends paid units). Runtime → Change runtime type → **T4 GPU**.
2. 🔑 Secrets (left bar): `HF_TOKEN` and `KAGGLE_API_TOKEN` (values from `clausd/.env`), Notebook access **on**.
3. Runtime → **Run all**. The link appears after ~3 minutes; Aman/Rohit work first, the other engines come online over the next 10-15 minutes (first run only installs; the status box on the link's page shows each engine).
4. Give the link to Claude, or paste it into Voiceover Studio → Cloud GPU & Keys → colab_gradio_url (saved for every browser).

The server code is downloaded fresh from your private Kaggle dataset each run, so fixes arrive without re-uploading this notebook.

In [ ]:
# CELL 1 - keys, server code, engines (first run ~10 min; the link in Cell 2 starts working for Aman after ~3 min)
import os, glob, subprocess
os.environ["USE_TF"] = "0"; os.environ["TRANSFORMERS_NO_TF"] = "1"
from google.colab import userdata
for k in ("HF_TOKEN", "KAGGLE_API_TOKEN"):
    try:
        os.environ[k] = userdata.get(k) or ""
    except Exception:
        os.environ[k] = ""
    print(k, "found" if os.environ[k] else "MISSING - add it in the key icon (Secrets) and run again")
!nvidia-smi -L
!pip install -q "kaggle>=1.8"
!kaggle datasets download alek6835210/fw-colab-voice-server -p /content/fwsrv --unzip --force -q
print("server files:", sorted(os.listdir("/content/fwsrv")))
# the Claude-trained FavoriteWorld voice (from the Kaggle training notebook output)
!kaggle kernels output alek6835210/fw-voice-claude-train -p /content/fwtrained --file-pattern "fw_claude_voice/model/[^/]+$" -q
hits = glob.glob("/content/fwtrained/**/fw_claude_voice/model/config.json", recursive=True)
os.environ["FW_TRAINED_DIR"] = os.path.dirname(hits[0]) if hits else ""
print("trained voice:", os.environ["FW_TRAINED_DIR"] or "not found")
# engines install side by side in the background; Cell 2 starts the link right away
subprocess.Popen("bash /content/fwsrv/setup_all.sh > /content/fwsetup.log 2>&1", shell=True)
print("installing engines in the background - log: /content/fwsetup.log")

In [ ]:
# CELL 2 - start the link (keep this tab open). Engines come online as their installs finish.
import time, os
for _ in range(180):          # wait for the Parler venv (needed first)
    if os.path.exists("/content/fwvenvs/parler/.done"):
        break
    time.sleep(5)
!tail -n 20 /content/fwsetup.log
!cd /content/fwsrv && python front.py